# 🤗 HuggingFace Open-Source Q&A — Mistral 7B

מחברת זו מורידה את **Mistral-7B-Instruct** מ-HuggingFace ומאפשרת לשאול שאלות ולקבל תשובות.

| תכונה | פרטים |
|--------|--------|
| מודל | `mistralai/Mistral-7B-Instruct-v0.2` |
| פרמטרים | 7B |
| Quantization | 4-bit (bitsandbytes) — ~4GB VRAM |
| רישיון | Apache 2.0 |
| GPU מינימלי | T4 (Colab חינמי) |

> **הפעל GPU לפני הרצה:** Runtime → Change runtime type → T4 GPU

## 1. התקנת ספריות

In [1]:
!pip install transformers torch accelerate bitsandbytes sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00


## 2. בחירת מודל

| מודל | פרמטרים | VRAM (4-bit) | הערות |
|------|---------|--------------|-------|
| `mistralai/Mistral-7B-Instruct-v0.2` | 7B | ~4GB | ברירת מחדל, מצוין |
| `mistralai/Mixtral-8x7B-Instruct-v0.1` | 47B MoE | ~24GB | דורש Colab Pro+ |
| `microsoft/Phi-3-mini-4k-instruct` | 3.8B | ~2.5GB | קל יותר, T4 בנוחות |
| `google/gemma-2b-it` | 2B | ~1.5GB | המהיר ביותר |

In [2]:

# MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
# MODEL_NAME = "google/gemma-2b-it"
# MODEL_NAME = "mistralai/Mixtral-8x7B-Instruct-v0.1"  # דורש Colab Pro+

## 3. טעינת המודל עם 4-bit Quantization

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
assert torch.cuda.is_available(), "GPU not found! Enable GPU: Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Loading model: {MODEL_NAME} (4-bit quantization)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"✅ Model loaded! VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-80GB
Loading model: mistralai/Mistral-7B-Instruct-v0.2 (4-bit quantization)...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Model loaded! VRAM used: 4.1 GB


## 4. פונקציית שאלה-תשובה

In [4]:
def ask(question: str, max_new_tokens: int = 512, temperature: float = 0.7) -> str:
    messages = [{"role": "user", "content": question}]
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = f"[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][input_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 5. שאלות לדוגמה

In [5]:
example_questions = [
    "What is the capital of France?",
    "Explain quantum entanglement in simple terms.",
    "What are the main differences between Python and JavaScript?",
    "מה הבירה של ישראל?",
]

for q in example_questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("-" * 60)

Q: What is the capital of France?
A: The capital city of France is Paris. Paris is one of the most visited cities in the world and is known for its iconic landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and Arc de Triomphe. It is also home to numerous cafes, restaurants, and shops, making it a popular destination for tourists and locals alike. The city is located in the northern part of France, in the region of Île-de-France.
------------------------------------------------------------
Q: Explain quantum entanglement in simple terms.
A: Quantum entanglement is a phenomenon in quantum physics where two or more particles become connected in such a way that the state of one particle is directly influenced by the state of the other, no matter how far apart they are. This connection is so strong that measuring the state of one particle instantly determines the state of the other particle, even if they are light-years apart. It's as if they share a single existence a

## 6. ממשק אינטראקטיבי

הרץ את התא הבא ושאל שאלות חופשיות. כתוב `exit` כדי לסיים.

In [7]:
print("🤖 Mistral Q&A Bot ready! Type 'exit' to quit.\n")
while True:
    question = input("Your question: ").strip()
    if not question:
        continue
    if question.lower() in ("exit", "quit", "q", "יציאה"):
        print("Goodbye!")
        break
    print(f"Answer: {ask(question)}\n")

🤖 Mistral Q&A Bot ready! Type 'exit' to quit.

Your question: אתה מכיר את הספר - מסילת ישרים
Answer: לא אני מכיר ספרים, אני עזור לתרגיל שאלות ומסגרת מידע. הספר שם מסילת ישרים מאורח בידידית לזכריה גרשוні, והוא נוצר בשנת 1943. הספר שוותאי לשפה עברית ויהיה מסוגל לשמור את זכורותי של החולים והעולה בתשובה וספק להשאות על נפשותם ועל החיים שלהם בתוך המחלה. הספר נהגש לאוצר המספרים השני בישראל ונתמכה בשיעורים בכל ישוב ומקום ביסוד ובישראל. יש לו בהחלטה פורסם על ידידיה הישראלית, והוא מסופק לבנים יותר מ 18 שנים ולמורים בשכולים. אם תרצה ללמוד על מסילת ישראל ניהול יותר מעל מספר שוב, אנא אז ארחיז את אימרתי ואל תרדפיני מנהל

Your question: יש לך tools לקבל תוכן מהאינטרנט ?
Answer: I'm an AI language model and don't have the ability to directly receive content from the internet or any other external source. I can only process and generate text based on the instructions or information provided to me. However, I can help you understand how to download or access content from the internet using various tools

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_8104/3779150805.py", line 3, in <cell line: 0>
    question = input("Your question: ").strip()
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 1177, in raw_input
    return self._input_request(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 1219, in _input_request
    raise KeyboardInterrupt("Interrupted by user") from None
KeyboardInterrupt: Interrupted by user

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^

TypeError: object of type 'NoneType' has no len()